# N1 — MAESTRO data, EDA and labels

Download official MAESTRO v3.0.0. Debug mode uses a small subset. Keep official splits.

In [1]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd() 
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from piano_transcriber.data import download_maestro
from piano_transcriber.features import load_audio,log_mel,augment_audio
from piano_transcriber.labels import midi_to_roll
csv=download_maestro(CFG['data']['root']); meta=pd.read_csv(csv); display(meta.head());display(meta.groupby('split').size())

FileNotFoundError: [Errno 2] No such file or directory: '/config.yaml'

In [ ]:
r=meta.iloc[0]; base=Path(csv).parent
y,sr=load_audio(base/r.audio_filename,CFG['features']['sample_rate'])
mel=log_mel(y,sr,CFG['features']['n_fft'],CFG['features']['hop_length'],CFG['features']['n_mels'],CFG['features']['fmin'],CFG['features']['fmax'])
roll,onset=midi_to_roll(base/r.midi_filename,mel.shape[1],sr,CFG['features']['hop_length'],21,108)
print(y.shape,mel.shape,roll.shape,'onsets',onset.sum())

In [ ]:
plt.figure(figsize=(12,4));plt.imshow(mel,origin='lower',aspect='auto');plt.title('Log-mel');plt.show()
plt.figure(figsize=(12,4));plt.imshow(roll.T,origin='lower',aspect='auto');plt.title('Aligned MIDI piano roll');plt.xlabel('Frame');plt.ylabel('Pitch index');plt.show()
plt.bar(np.arange(88)+21,roll.sum(0));plt.title('Pitch occupancy');plt.show()

Augmentation is deliberately conservative (gain/noise). Add time stretch or pitch shift only after validating alignment effects. Place own WAV/MP3 recordings in `data/own_recordings`; add aligned MIDI to calculate objective metrics.